In [5]:
pip install rdkit

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

In [148]:
import warnings
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

# Suppress deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

def calculate_morgan_fingerprint(smiles, radius=2, nBits=1024):
    try:
        molecule = Chem.MolFromSmiles(smiles)
        
        if molecule is None:
            return None
        
        # Generate Morgan fingerprint as a bit vector
        fingerprint = AllChem.GetMorganFingerprintAsBitVect(molecule, radius=radius, nBits=nBits)
        
        # Convert the bit vector to a bit string
        return fingerprint.ToBitString()
    
    except Exception as e:
        print(f"Error processing SMILES '{smiles}': {e}")
        return None

def process_csv(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    if 'SMILES' not in df.columns:
        raise ValueError("Input CSV must contain a 'SMILES' column.")
    
    # Apply the fingerprint calculation
    df['morgan_fingerprint'] = df['SMILES'].apply(calculate_morgan_fingerprint)
    # Save the output CSV
    df.to_csv(output_csv, index=False)

# Example usage
input_csv = '1000_generated_ligands-1.csv' 
output_csv = '1000_generated_ligands-1_final_fingerprint.csv'  

process_csv(input_csv, output_csv)


[19:29:12] Can't kekulize mol.  Unkekulized atoms: 13 14 15 16 17
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12] Can't kekulize mol.  Unkekulized atoms: 15 16 17 24 25
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12] Can't kekulize mol.  Unkekulized atoms: 15 16 17 24 25
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12] Can't kekulize mol.  Unkekulized atoms: 4 5 6 13 14
[19:29:12] Can't kekulize mol.  Unkekulized atoms: 16 17 18 25 26
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12] Can't kekulize mol.  Unkekulized atoms: 19 20 21 28 29
[19:29:12] Can't kekulize mol.  Unkekulized atoms: 26 27 28 35 36
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12] DEPRECATION WARNING: please use MorganGenerator
[19:29:12]

In [149]:
df = pd.read_csv('1000_generated_ligands-1_final_fingerprint.csv')
df


,SMILES,morgan_fingerprint
0,P(c1c(N(C)C)cccc1N(C)C)(c1ccnn1)(c1cccc2c1[P@@...,NaN
1,P(c1cccc2c1[P@@](C(C)(C)C)CO2)(c1c(OC)cccc1OC)...,0000000000000001000000100000000001000000000001...
2,P(c1cc(C(C)(C)C)c(OC)cc1OC)(c1cc(-c2ccccc2)nn1...,NaN
3,P(c1cc(OC(C)C)c(C(C)(C)C)cc1)(c1c(N(C)C)cccc1N...,0110100000000001000000000000000001100000000001...
4,P(N1CCCCC1)(c1ccccc1N(C)C)(c1ccccc1P(c1ccccc1)...,0010100000000000000000000000000001000000000001...
...,...,...
1017,P(C1[N@]2C[N@@]3CP1C[N@](C2)C3)(c1ccnn1)(c1ccc...,NaN
1018,P(c1ccc(-c2cccc3c2PCO3)cc1)(c1ccccc1)(c1cc(C(C...,0000000000000001010000000000000001000000000001...
1019,P(c1ccnn1)(c1ccncc1)(c1ccc(N(C)C)cc1N(C)C),NaN
1020,P(c1c(P(c2cc(C)cc(C)c2)c2cc(C)cc(C)c2)cc(OC)nc...,0000000000000000000000000000000001000000000001...


In [150]:
# df = pd.read_csv('lab_edit_smiles.csv')
# df = df.drop(['ID'], axis=1)
# df = df.rename(columns={'Yield Cross-Coupling': 'yield'})
# df.head()


In [151]:

# Select rows that contain at least one NaN value
df_nan_rows = df[df.isna().any(axis=1)]

# Save to CSV file
df_nan_rows.to_csv("nan_rows.csv", index=False)

print("CSV file 'nan_rows.csv' saved successfully!")


CSV file 'nan_rows.csv' saved successfully!


In [140]:
# Step 1: Replace None, empty strings, and 'NaN' with np.nan (Avoiding inplace warning)
df['morgan_fingerprint'] = df['morgan_fingerprint'].replace([None, '', 'NaN'], np.nan)

# Step 2: Drop rows where 'morgan_fingerprint' has NaN values
df = df.dropna(subset=['morgan_fingerprint'])

# Step 3: Ensure 'morgan_fingerprint' is treated as a string
df['morgan_fingerprint'] = df['morgan_fingerprint'].astype(str)


C:\Users\UTKARSH GUPTA\AppData\Local\Temp\ipykernel_20080\841961094.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['morgan_fingerprint'] = df['morgan_fingerprint'].astype(str)


In [141]:
len(df)

620

In [142]:
# Step 4: Function to split fingerprint into separate columns
def split_bits(fingerprint):
    return list(fingerprint)  # Converts string into a list of bits

# Apply function correctly without changing row count
bit_columns = df['morgan_fingerprint'].apply(split_bits).apply(pd.Series)

# Rename columns correctly
bit_columns.columns = [f'F_{i+1}' for i in range(bit_columns.shape[1])]

# Step 5: Concatenate original DataFrame with new bit columns
final_df = pd.concat([df.reset_index(drop=True), bit_columns], axis=1)


In [143]:
len(final_df)

858

In [144]:
final_df = final_df.dropna()

In [146]:
input = final_df.copy()
del input['morgan_fingerprint']
del input['SMILES']
#del input['yield']
input.head()

,F_1,F_2,F_3,F_4,F_5,F_6,F_7,F_8,F_9,F_10,...,F_1015,F_1016,F_1017,F_1018,F_1019,F_1020,F_1021,F_1022,F_1023,F_1024
1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,0,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,1,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [147]:
len(final_df)

382

In [117]:
for column in input.columns:
    if input[column].dtype == 'object':
        input[column] = pd.to_numeric(input[column], errors='coerce')
# Check the new data types
print(input.dtypes)

F_1       int64
F_2       int64
F_3       int64
F_4       int64
F_5       int64
          ...  
F_1020    int64
F_1021    int64
F_1022    int64
F_1023    int64
F_1024    int64
Length: 1024, dtype: object


In [118]:
pip install xgboost


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [119]:
import pandas as pd
import xgboost as xgb
import os
import logging

In [120]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [121]:
input_df = input.copy()
final_df = input_df.copy()

In [122]:
models_directory = 'models_xgboost_bayes_val_cv'  
model_files = [f for f in os.listdir(models_directory) if f.endswith('.pkl')]
logging.info(f"Found {len(model_files)} models.")

2025-03-12 18:58:49,448 - INFO - Found 190 models.


In [123]:
prediction_dfs = []

for model_file in model_files:
    model_path = os.path.join(models_directory, model_file)
    logging.info(f"Loading model: {model_file}...")
 
    try:
        xgb_model = xgb.Booster()
        xgb_model.load_model(model_path)
        logging.info(f"Successfully loaded model: {model_file}")
    except Exception as e:
        logging.error(f"Error loading model {model_file}: {e}")
        continue  
        
    dmatrix = xgb.DMatrix(input_df)

    logging.info(f"Making predictions with model: {model_file}...")
    predictions = xgb_model.predict(dmatrix)

    model_name = model_file.replace('.pkl', '')  
    
    predictions_df = pd.DataFrame(predictions, columns=[model_name])
    prediction_dfs.append(predictions_df)
    logging.info(f"Predictions added for model: {model_file}")

final_predictions_df = pd.concat(prediction_dfs, axis=1)

predictions_output_file = 'High_ligands_final_final_predictions_output.csv'
final_predictions_df.to_csv(predictions_output_file, index=False)
logging.info(f"Predictions CSV with {len(model_files)} columns has been saved as '{predictions_output_file}'.")


2025-03-12 18:58:49,474 - INFO - Loading model: model_dipolemoment_boltz.pkl...
2025-03-12 18:58:49,503 - INFO - Successfully loaded model: model_dipolemoment_boltz.pkl
2025-03-12 18:58:50,657 - INFO - Making predictions with model: model_dipolemoment_boltz.pkl...
2025-03-12 18:58:50,673 - INFO - Predictions added for model: model_dipolemoment_boltz.pkl
2025-03-12 18:58:50,677 - INFO - Loading model: model_dipolemoment_delta.pkl...
2025-03-12 18:58:50,698 - INFO - Successfully loaded model: model_dipolemoment_delta.pkl
2025-03-12 18:58:51,810 - INFO - Making predictions with model: model_dipolemoment_delta.pkl...
2025-03-12 18:58:51,821 - INFO - Predictions added for model: model_dipolemoment_delta.pkl
2025-03-12 18:58:51,828 - INFO - Loading model: model_dipolemoment_max.pkl...
2025-03-12 18:58:51,857 - INFO - Successfully loaded model: model_dipolemoment_max.pkl
2025-03-12 18:58:52,858 - INFO - Making predictions with model: model_dipolemoment_max.pkl...
2025-03-12 18:58:52,867 - INF

In [124]:
output = pd.read_csv('High_ligands_final_final_predictions_output.csv')
output.head()

,model_dipolemoment_boltz,model_dipolemoment_delta,model_dipolemoment_max,model_dipolemoment_min,model_dipolemoment_vburminconf,model_efgtens_xx_P_boltz,model_efgtens_yy_P_boltz,model_efgtens_zz_P_boltz,model_efg_amp_P_boltz,model_E_oxidation_boltz,...,model_vbur_ratio_vbur_vtot_boltz,model_vbur_vbur_boltz,model_vbur_vbur_delta,model_vbur_vbur_max,model_vbur_vbur_min,model_vbur_vbur_vburminconf,model_vbur_vtot_boltz,model_vmin_r_boltz,model_vmin_vmin_boltz,model_volume_boltz
0,3.601101,2.746757,3.848183,1.653122,3.205452,-1.071156,-0.498768,1.554530,1.887761,0.256783,...,0.220631,98.104920,14.106120,102.835390,82.80130,77.535370,422.66220,1.811474,-0.065044,532.77014
1,3.246665,2.334495,3.424423,1.779119,3.328659,-0.936143,-0.445119,1.459685,1.805756,0.263688,...,0.162947,86.942070,44.516056,106.617520,61.98695,59.169624,549.79650,1.802640,-0.063585,650.82434
2,3.659574,1.829604,2.775361,1.748480,3.053771,-0.974400,-0.494986,1.544453,1.865017,0.260458,...,0.232910,96.745445,14.021627,101.890366,82.45664,77.446250,505.30374,1.811892,-0.063683,664.99713
3,1.702560,2.888522,3.489856,0.961759,2.456537,-0.838840,-0.601655,1.424141,1.751057,0.255795,...,0.218195,101.473580,58.310913,121.368340,56.29661,56.541195,437.52540,1.803918,-0.065115,538.28253
4,0.706851,1.961700,2.091641,0.769939,2.787268,-0.846619,-0.601787,1.428497,1.758927,0.250412,...,0.222260,107.695760,44.100914,116.998710,77.85579,77.615710,479.07794,1.788780,-0.067359,584.05963


In [125]:
output['input'] = df['morgan_fingerprint']
output.head()

,model_dipolemoment_boltz,model_dipolemoment_delta,model_dipolemoment_max,model_dipolemoment_min,model_dipolemoment_vburminconf,model_efgtens_xx_P_boltz,model_efgtens_yy_P_boltz,model_efgtens_zz_P_boltz,model_efg_amp_P_boltz,model_E_oxidation_boltz,...,model_vbur_vbur_boltz,model_vbur_vbur_delta,model_vbur_vbur_max,model_vbur_vbur_min,model_vbur_vbur_vburminconf,model_vbur_vtot_boltz,model_vmin_r_boltz,model_vmin_vmin_boltz,model_volume_boltz,input
0,3.601101,2.746757,3.848183,1.653122,3.205452,-1.071156,-0.498768,1.554530,1.887761,0.256783,...,98.104920,14.106120,102.835390,82.80130,77.535370,422.66220,1.811474,-0.065044,532.77014,0000000000000000000000000000000001000000000101...
1,3.246665,2.334495,3.424423,1.779119,3.328659,-0.936143,-0.445119,1.459685,1.805756,0.263688,...,86.942070,44.516056,106.617520,61.98695,59.169624,549.79650,1.802640,-0.063585,650.82434,0010100000000000000000000000000001000000000101...
2,3.659574,1.829604,2.775361,1.748480,3.053771,-0.974400,-0.494986,1.544453,1.865017,0.260458,...,96.745445,14.021627,101.890366,82.45664,77.446250,505.30374,1.811892,-0.063683,664.99713,0000000000000000000000000000000001000000000101...
3,1.702560,2.888522,3.489856,0.961759,2.456537,-0.838840,-0.601655,1.424141,1.751057,0.255795,...,101.473580,58.310913,121.368340,56.29661,56.541195,437.52540,1.803918,-0.065115,538.28253,0010100000000000000000000000000101000000000001...
4,0.706851,1.961700,2.091641,0.769939,2.787268,-0.846619,-0.601787,1.428497,1.758927,0.250412,...,107.695760,44.100914,116.998710,77.85579,77.615710,479.07794,1.788780,-0.067359,584.05963,0010100000000001000000000000000101000000000001...


In [126]:
last_column = output.iloc[:, -1] 
output = output.iloc[:, :-1]  
output.insert(0, last_column.name, last_column) 
output.head()

,input,model_dipolemoment_boltz,model_dipolemoment_delta,model_dipolemoment_max,model_dipolemoment_min,model_dipolemoment_vburminconf,model_efgtens_xx_P_boltz,model_efgtens_yy_P_boltz,model_efgtens_zz_P_boltz,model_efg_amp_P_boltz,...,model_vbur_ratio_vbur_vtot_boltz,model_vbur_vbur_boltz,model_vbur_vbur_delta,model_vbur_vbur_max,model_vbur_vbur_min,model_vbur_vbur_vburminconf,model_vbur_vtot_boltz,model_vmin_r_boltz,model_vmin_vmin_boltz,model_volume_boltz
0,0000000000000000000000000000000001000000000101...,3.601101,2.746757,3.848183,1.653122,3.205452,-1.071156,-0.498768,1.554530,1.887761,...,0.220631,98.104920,14.106120,102.835390,82.80130,77.535370,422.66220,1.811474,-0.065044,532.77014
1,0010100000000000000000000000000001000000000101...,3.246665,2.334495,3.424423,1.779119,3.328659,-0.936143,-0.445119,1.459685,1.805756,...,0.162947,86.942070,44.516056,106.617520,61.98695,59.169624,549.79650,1.802640,-0.063585,650.82434
2,0000000000000000000000000000000001000000000101...,3.659574,1.829604,2.775361,1.748480,3.053771,-0.974400,-0.494986,1.544453,1.865017,...,0.232910,96.745445,14.021627,101.890366,82.45664,77.446250,505.30374,1.811892,-0.063683,664.99713
3,0010100000000000000000000000000101000000000001...,1.702560,2.888522,3.489856,0.961759,2.456537,-0.838840,-0.601655,1.424141,1.751057,...,0.218195,101.473580,58.310913,121.368340,56.29661,56.541195,437.52540,1.803918,-0.065115,538.28253
4,0010100000000001000000000000000101000000000001...,0.706851,1.961700,2.091641,0.769939,2.787268,-0.846619,-0.601787,1.428497,1.758927,...,0.222260,107.695760,44.100914,116.998710,77.85579,77.615710,479.07794,1.788780,-0.067359,584.05963


In [133]:
first_column = output.iloc[:, 0]  
sorted_columns = sorted(output.columns[1:])  

final_lab_ligand = pd.concat([first_column, output[sorted_columns]], axis=1)


final_lab_ligand['yield'] = final_df['yield']
final_lab_ligand['SMILES'] = final_df['SMILES']
final_lab_ligand.head()


,input,model_E_oxidation_boltz,model_E_reduction_boltz,model_E_solv_cds_boltz,model_E_solv_elstat_boltz,model_E_solv_total_boltz,model_Pint_P_int_boltz,model_Pint_P_max_boltz,model_Pint_P_min_boltz,model_Pint_dP_boltz,...,model_vbur_vbur_delta,model_vbur_vbur_max,model_vbur_vbur_min,model_vbur_vbur_vburminconf,model_vbur_vtot_boltz,model_vmin_r_boltz,model_vmin_vmin_boltz,model_volume_boltz,yield,SMILES
0,0000000000000000000000000000000001000000000101...,0.256783,0.023705,-4.816398,-9.268672,-15.162958,20.015041,36.363710,12.033858,3.981717,...,14.106120,102.835390,82.80130,77.535370,422.66220,1.811474,-0.065044,532.77014,91,COC1=CC=C(N2C(P(C(C)(C)C)C(C)(C)C)=CC=N2)C(OC)=C1
1,0010100000000000000000000000000001000000000101...,0.263688,0.020393,-9.382906,-7.951180,-15.298017,20.243896,40.929077,12.579647,4.573813,...,44.516056,106.617520,61.98695,59.169624,549.79650,1.802640,-0.063585,650.82434,46,COC1=CC(OC)=C(C(C)(C)C)C=C1N2C(P(C3CCCCC3)C4CC...
2,0000000000000000000000000000000001000000000101...,0.260458,0.020929,-6.768596,-8.390507,-15.078186,20.086765,39.388270,12.579207,4.014885,...,14.021627,101.890366,82.45664,77.446250,505.30374,1.811892,-0.063683,664.99713,93,CC(P(C1=CC=NN1C2=CC(C(C)(C)C)=C(OC)C=C2OC)C(C)...
3,0010100000000000000000000000000101000000000001...,0.255795,0.022808,-9.089910,-8.239961,-16.703497,20.002974,38.729824,12.072245,4.542177,...,58.310913,121.368340,56.29661,56.541195,437.52540,1.803918,-0.065115,538.28253,33,COC1=C(C2=C(P(C3CCCCC3)C4CCCCC4)C=CC=C2)C=CC(O...
4,0010100000000001000000000000000101000000000001...,0.250412,0.023673,-9.482734,-7.935542,-17.410470,20.461084,38.761520,13.001906,4.536157,...,44.100914,116.998710,77.85579,77.615710,479.07794,1.788780,-0.067359,584.05963,44,CN(C1=C(C2=C(P(C3CCCCC3)C4CCCCC4)C=CC=C2)C(OC)...


In [134]:
len(final_lab_ligand)

7

In [135]:
count = final_lab_ligand.isna().any(axis=1).sum()
count

0

In [136]:
final_lab_ligand.to_csv('High_ligands_final_final_features_output.csv', index=False)


In [1]:
import pandas as pd 

In [16]:

df_n = pd.read_csv("new_smiles.csv")
df_n.head()


,SMILES,yield,morgan_fingerprint
0,CN(C1=C(C2=C(P(C3CCCCC3)C4CCCCC4)C=CC=C2)C=CC(...,20,0010100000000000000000000000000101000000000001...
1,CN(C1=C(C2=C(P(C3=CC=CC=C3)C4=CC=CC=C4)C=CC=C2...,10,0000000000000000000000000000000001000000000001...
2,CC(P(C(C=CC=C1)=C1C2=C(N(C)C)C=C(N(C)C)C=C2)C(...,0,0100000000000000000000000000000001000000001001...
3,COC1=CC=C(N2C(P(C3CCCCC3)C4CCCCC4)=CC=N2)C(OC)=C1,0,0010100000000000000000000000000001000000000101...
4,COC1=CC=C(N2C(P(C(C)(C)C)C(C)(C)C)=CC=N2)C(OC)=C1,62,0000000000000000000000000000000001000000000101...


In [17]:
df.head()

,SMILES,yield,morgan_fingerprint
0,CCP(CC)CC,0,0000000000000000000000000000000001000000000001...
1,CP(C)C,0,0000000000000000000000000000000001000000000001...
2,[N@@]1(C[N@]2C3)CP3C[N@](C2)C1,0,0000000000000000000000000000000000000000000000...
3,CC(CCC)P(C1CCCCC1)C2CCCCC2,0,0110100000000000000000000000000001000000000001...
4,CCCCP([C@]1(C[C@H]2C3)C[C@@H](C2)C[C@@H]3C1)[C...,0,0000000000000000000000000000000001001000000001...


In [27]:
df1 = pd.concat([df, df_n], ignore_index=True)
df1 = df1.rename(columns={'morgan_fingerprint': 'input'})
df1.head()

,SMILES,yield,input
0,CCP(CC)CC,0,0000000000000000000000000000000001000000000001...
1,CP(C)C,0,0000000000000000000000000000000001000000000001...
2,[N@@]1(C[N@]2C3)CP3C[N@](C2)C1,0,0000000000000000000000000000000000000000000000...
3,CC(CCC)P(C1CCCCC1)C2CCCCC2,0,0110100000000000000000000000000001000000000001...
4,CCCCP([C@]1(C[C@H]2C3)C[C@@H](C2)C[C@@H]3C1)[C...,0,0000000000000000000000000000000001001000000001...


In [37]:
final = pd.read_csv("final_ligands.csv")

In [38]:
final.head()

,input,model_E_oxidation_boltz,model_E_reduction_boltz,model_E_solv_cds_boltz,model_E_solv_elstat_boltz,model_E_solv_total_boltz,model_Pint_P_int_boltz,model_Pint_P_max_boltz,model_Pint_P_min_boltz,model_Pint_dP_boltz,...,model_vbur_vbur_delta,model_vbur_vbur_max,model_vbur_vbur_min,model_vbur_vbur_vburminconf,model_vbur_vtot_boltz,model_vmin_r_boltz,model_vmin_vmin_boltz,model_volume_boltz,yield,index_column
0,0000000000000000000000000000000001000000000001...,0.303620,0.076862,-2.404362,-3.899914,-5.572245,16.438795,26.658260,11.354961,3.364319,...,11.970101,55.491146,44.365550,44.634552,178.65941,1.829504,-0.053807,215.21457,0,0
1,0000000000000000000000000000000001000000000001...,0.315292,0.079952,0.100458,-5.185666,-3.445140,15.564138,25.196466,11.504475,3.012862,...,3.678765,42.227734,42.226490,43.004677,117.04840,1.835588,-0.048619,148.69606,0,1
2,0000000000000000000000000000000000000000000000...,0.298519,0.066568,-1.781534,-4.142239,-6.172190,17.187168,27.688345,12.257298,3.036557,...,11.170786,57.019880,43.270283,40.860226,210.79329,1.892139,-0.042421,266.44974,0,2
3,0110100000000000000000000000000001000000000001...,0.282032,0.061337,-6.985048,-3.539417,-10.239269,18.203201,31.063185,11.856228,3.557869,...,24.009241,78.151146,53.715330,52.901450,297.37643,1.795672,-0.059711,341.08774,0,3
4,0000000000000000000000000000000001001000000001...,0.267603,0.056862,-9.630296,-4.263012,-13.797570,19.652014,31.407488,11.884287,3.658416,...,21.787224,73.715370,58.651604,58.704190,395.04953,1.783544,-0.066182,493.20935,0,4


In [39]:
len(final)

61

In [40]:
result = pd.merge(final, df1[['input', 'SMILES']], on='input', how='inner')
result.head()

,input,model_E_oxidation_boltz,model_E_reduction_boltz,model_E_solv_cds_boltz,model_E_solv_elstat_boltz,model_E_solv_total_boltz,model_Pint_P_int_boltz,model_Pint_P_max_boltz,model_Pint_P_min_boltz,model_Pint_dP_boltz,...,model_vbur_vbur_max,model_vbur_vbur_min,model_vbur_vbur_vburminconf,model_vbur_vtot_boltz,model_vmin_r_boltz,model_vmin_vmin_boltz,model_volume_boltz,yield,index_column,SMILES
0,0000000000000000000000000000000001000000000001...,0.303620,0.076862,-2.404362,-3.899914,-5.572245,16.438795,26.658260,11.354961,3.364319,...,55.491146,44.365550,44.634552,178.65941,1.829504,-0.053807,215.21457,0,0,CCP(CC)CC
1,0000000000000000000000000000000001000000000001...,0.315292,0.079952,0.100458,-5.185666,-3.445140,15.564138,25.196466,11.504475,3.012862,...,42.227734,42.226490,43.004677,117.04840,1.835588,-0.048619,148.69606,0,1,CP(C)C
2,0000000000000000000000000000000000000000000000...,0.298519,0.066568,-1.781534,-4.142239,-6.172190,17.187168,27.688345,12.257298,3.036557,...,57.019880,43.270283,40.860226,210.79329,1.892139,-0.042421,266.44974,0,2,[N@@]1(C[N@]2C3)CP3C[N@](C2)C1
3,0110100000000000000000000000000001000000000001...,0.282032,0.061337,-6.985048,-3.539417,-10.239269,18.203201,31.063185,11.856228,3.557869,...,78.151146,53.715330,52.901450,297.37643,1.795672,-0.059711,341.08774,0,3,CC(CCC)P(C1CCCCC1)C2CCCCC2
4,0000000000000000000000000000000001001000000001...,0.267603,0.056862,-9.630296,-4.263012,-13.797570,19.652014,31.407488,11.884287,3.658416,...,73.715370,58.651604,58.704190,395.04953,1.783544,-0.066182,493.20935,0,4,CCCCP([C@]1(C[C@H]2C3)C[C@@H](C2)C[C@@H]3C1)[C...


In [41]:
len(result)

67

In [33]:
result.to_csv('Lignads_smiles_descriptors.csv', index=False)